In [2]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/clean_modules_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/top_corr_module_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/module_projection_fxns.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: future


Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths


The following objects are masked from ‘package:data.table’:

    dcast, melt




In [3]:
mod_def <- "TopModPosFDR"

In [4]:
marker_source <- ""

marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_human.RDS")

## Prep data

In [ ]:
data_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed"

In [ ]:
bulk_expr <- fread("GTEx_cortex_counts_TMMF_SampleNetworks/All_02-25-12/GTEx_cortex_counts_TMMF_All_501_outliers_removed.csv", data.table=FALSE)
colnames(bulk_expr)[1] <- "Gene"

In [6]:
# bulk_expr <- fread("GTEx_cortex_counts_TMMF_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_SampleNetworks/All_02-59-24/GTEx_cortex_counts_TMMF_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_All_370_outliers_removed.csv", data.table=FALSE)
# colnames(bulk_expr)[1] <- "Gene"

In [7]:
# mean_expr <- data.frame(
#     Index=1:nrow(bulk_expr), 
#     Gene=bulk_expr[,1], 
#     Expr=rowMeans(bulk_expr[,-1])
# )

# mean_expr <- mean_expr %>%
#     group_by(Gene) %>%
#     slice_max(Expr, with_ties=FALSE)

# print(dim(mean_expr))

# bulk_expr <- bulk_expr[mean_expr$Index,]

## Select modules to remove

In [12]:
enrichment_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_mergeParam0.95_subsetCutoff10.171_Modules_MO_1271sets_enrichments_top_Qval_mods"

top_mods_df <- fread(paste0("data/enrichments/", enrichment_source, ".csv"), data.table=FALSE)

sets_to_remove <- c(
    "kelley_microglia", "kelley_oligo", "kelley_Mural", "kelley_Endothelial",
    "JAKEL_2019_IMMUNE", "kelley_neuron", "OLDHAM_CORTEX_MODULE_PURPLE",
    "OLDHAM_CORTEX_MODULE_ORANGE"
)

In [13]:
top_mods_df_subset <- top_mods_df[top_mods_df$Cell_type %in% sets_to_remove,]

In [ ]:
# enrichment_source1 <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_mergeParam0.95_subsetCutoff10.171_Modules_MO_1271sets_enrichments_top_Qval_mods"

# top_mods_df1 <- fread(paste0("data/enrichments/", enrichment_source, ".csv"), data.table=FALSE)

# sets_to_remove1 <- c(
#     "kelley_microglia", "kelley_oligo", "kelley_Mural", "kelley_Endothelial",
#     "JAKEL_2019_IMMUNE", "kelley_neuron", "OLDHAM_CORTEX_MODULE_PURPLE",
#     "OLDHAM_CORTEX_MODULE_ORANGE"
# )

# enrichment_source2 <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_mergeParam0.95_subsetCutoff10.171_Modules_MSigDB_6916sets_enrichments_top_Qval_mods"

# top_mods_df2 <- fread(paste0("data/enrichments/", enrichment_source, ".csv"), data.table=FALSE)

# sets_to_remove2 <- "" 

# sets_to_remove <- c(sets_to_remove1, sets_to_remove2)
# top_mods_df <- rbind(top_mods_df1, top_mods_df2)
# top_mods_df_subset <- top_mods_df[top_mods_df$Cell_type %in% sets_to_remove,]

In [15]:
exclude <- unlist(marker_genes_list[2:23]) # Don't remove genes associated with cell types of interest

expr_filtered <- remove_mod_genes(bulk_expr, top_mods_df_subset, mod_def, exclude) 

dim(expr_filtered)

[1] "16523 genes to be removed"


[1] 42147   502

## Save

In [ ]:
outdir <- paste0("data/cleaned/", mod_def)

if (!dir.exists(outdir)) {
    dir.create(outdir, recursive=TRUE)
}

filename <- paste0(outdir, "/", data_source, "_", nrow(expr_filtered), "genes_cleaned.csv")

fwrite(expr_filtered, file=filename)

## // Select modules to regress

In [ ]:
# # Get genes for each top module, and intersect with marker genes; which modules are these genes appearing in?

# marker_genes <- unlist(marker_genes_list[2:23])

# # inds <- c()
# overlap_genes <- vector(mode="list", length=nrow(top_mods_df_subset))
# names(overlap_genes) <- paste(top_mods_df_subset$Cell_type, top_mods_df_subset$Qval)

# for (i in 1:nrow(top_mods_df_subset)) {
#     mod <- top_mods_df_subset$Module[i]
#     kME_path <- top_mods_df_subset$kME_path[i]
#     mod_genes <- get_mod_genes(kME_path, mod, mod_def)
#     set_overlap_genes <- lapply(marker_genes_list, function(x) intersect(mod_genes, x))
#     if (length(set_overlap_genes) > 0) {
#       overlap_genes[[i]] <- set_overlap_genes 
#     }
# }

In [1]:
# enrichment_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_mergeParam0.95_subsetCutoff10.171_Modules_MO_1271sets_enrichments_top_Qval_mods"

# top_mods_df <- fread(paste0("data/enrichments/", enrichment_source, ".csv"), data.table=FALSE)

# top_mods_df %>%
#     arrange(Qval) %>%
#     select(Cell_type, Qval) %>%
#     head()

# # Make sure expression data and ME data samples match
# sets_to_regress <- c("ZHONG_PFC_MAJOR_TYPES_OPC", "GOBP_INTERLEUKIN_6_PRODUCTION")

# top_mods_df_subset <- top_mods_df[top_mods_df$Cell_type %in% sets_to_regress,]
# mod_eigs_df <- get_mod_eigs(top_mods_df_subset)

# mod_eigs_df <- mod_eigs_df[match(colnames(bulk_expr)[-1], mod_eigs_df$Sample),]

# all.equal(mod_eigs_df$Sample, colnames(bulk_expr)[-1])

# cor(mod_eigs_df[,-1])

# X <- model.matrix(~ ., data=mod_eigs_df[,-1])

# fit <- lmFit(bulk_expr[,-1], X)

# fitted_vals <- fit$coefficients %*% t(X)
# resid_mat <- bulk_expr[,-1] - fitted_vals
# expr_adj <- resid_mat # + rowMeans(expr[,-1])  # genes x sample

# bulk_expr_regr <- data.frame(Gene=bulk_expr[,1], expr_adj)
# bulk_expr_regr[1:5, 1:5]

ERROR: Error in fread(paste0("data/enrichments/", enrichment_source, ".csv"), : could not find function "fread"
